# **Versión 3: Dataset desbalanceado - CodeBERT**

Transformer preentrenado para NL y PL:
* 12 capas profundas
* Embeddings de 768 dimensiones
* Entrada máxima de 512 tokens

In [ ]:
!pip install transformers[torch] datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

# Configuración de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Montar Drive
drive.mount("/content/gdrive")
%cd "/content/gdrive/MyDrive/Desarrollo de aplicaciones avanzadas de ciencias computacionales"

# Carga del dataset
df_raw = pd.read_csv('df_combined.csv')

Mounted at /content/gdrive
/content/gdrive/MyDrive/Desarrollo de aplicaciones avanzadas de ciencias computacionales


## 1. Split inicial

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df_raw['code'], df_raw['target'], test_size=0.2, random_state=42, stratify=df_raw['target']
)

## 2. Tokenización y Clase Dataset

In [ ]:
model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class CodeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        # Ensure all texts are strings
        processed_texts = [str(text) for text in texts]
        self.encodings = tokenizer(processed_texts, truncation=True, padding=True, max_length=max_len)
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = CodeDataset(X_train_raw, y_train, tokenizer)
test_dataset = CodeDataset(X_test_raw, y_test, tokenizer)

## 3. Entrenamiento

In [ ]:
from torch import nn
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification

class BalancedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Cargar modelo pre-entrenado
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir='./logs',
)

trainer = BalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,0.201625,0.213635
2,0.117582,0.210255
3,0.081328,0.228267


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=3579, training_loss=0.14640390356681507, metrics={'train_runtime': 3374.5807, 'train_samples_per_second': 8.485, 'train_steps_per_second': 1.061, 'total_flos': 7533395737067520.0, 'train_loss': 0.14640390356681507, 'epoch': 3.0})

## 4. Evaluación de Resultados

In [ ]:
class CodeBERTWrapper:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.model.eval()

    def predict(self, texts):
        preds = []
        # Procesar en pequeños lotes para no agotar la memoria
        for i in range(0, len(texts), 16):
            batch = list(texts)[i:i+16]
            # Ensure all items in the batch are strings before tokenizing
            processed_batch = [str(text) for text in batch]
            inputs = self.tokenizer(processed_batch, truncation=True, padding=True, max_length=512, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = self.model(**inputs)
                batch_preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                preds.extend(batch_preds)
        return np.array(preds)

# Instanciar el wrapper
model_codebert = CodeBERTWrapper(model, tokenizer)

# Tu función de evaluación original
def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)

    print(f"Accuracy {model_name}:", accuracy_score(y_test, y_pred))
    print(f"\nMatriz de confusión {model_name}:")
    print(confusion_matrix(y_test, y_pred))
    print(f"\nReporte {model_name}:")
    print(classification_report(y_test, y_pred))

# Ejecución de la evaluación
evaluate_model(model_codebert, X_test_raw, y_test, model_name="CodeBERT")

Accuracy CodeBERT: 0.9522212908633697

Matriz de confusión CodeBERT:
[[1089   99]
 [  15 1183]]

Reporte CodeBERT:
              precision    recall  f1-score   support

           0       0.99      0.92      0.95      1188
           1       0.92      0.99      0.95      1198

    accuracy                           0.95      2386
   macro avg       0.95      0.95      0.95      2386
weighted avg       0.95      0.95      0.95      2386

